# 05. A production-shaped pipeline: SQL features, a hand-derived model, scores written back

**What this notebook does:** takes the pieces derived in notebooks 01 to 03 and arranges them the way
a scoring system runs in practice, on the Medicare provider data (CMS billing plus OIG exclusions,
sampled into `data/ml_derivations.sqlite` from the Medicare fraud project):

1. **Extract** the raw features with SQL
2. **Engineer** peer-adjusted features
3. **Train** a regularized logistic regression, implemented from the derivations, with class weights
4. **Score** every provider and **write the scores back** to the database as a table
5. **Serve** a top-$k$ worklist with a SQL query, and score a brand-new provider from saved weights

Every model line in this notebook is one from notebooks 02 and 03. Nothing is imported from scikit-learn
except the metric used to check the result.

## The one new piece of math: class weights

With 68 positives in 20,000, plain log loss lets the model ignore positives. Weighted log loss
multiplies each row's term by a weight $c_{y_i}$, with the rare class weighted up:

$$L = -\frac{1}{n}\sum_i c_{y_i}\Big[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\Big] + \lambda\|w\|_2^2
\qquad
c_1 = \frac{n}{2 n_1}, \; c_0 = \frac{n}{2 n_0}$$

The gradient is the same as before with each row's residual scaled by its weight, plus the ridge term:

$$\nabla_w L = \frac{1}{n}X^{\top}\big(c \odot (p - y)\big) + 2\lambda w$$

where $c \odot (p - y)$ means multiply row by row.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DB = "C:/Users/palla/OneDrive/Documents/Coding Projects/cheatsheets_and_templates/ML_derivations/data/ml_derivations.sqlite"
con = sqlite3.connect(DB)
def q(sql):
    """Run SQL against the project database and return a DataFrame."""
    return pd.read_sql(sql, con)

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#9a9a94"

## 1. Extract: raw features by SQL

**What this cell does:** pulls the 16 raw billing features, the specialty, and the label. The query
also computes two things SQL is good at: the count of providers per specialty, and a flag for
specialties too small to be their own peer group.

In [2]:
feature_columns = ["srvcs_per_bene", "pymt_per_bene", "chrg_to_alowd", "drug_pymt_share", "risk_score",
                   "pymt_per_bene_per_risk", "pct_under65", "pct_dual", "avg_age", "n_distinct_codes",
                   "tot_benes", "sameday_repeat_ratio", "em_high_share", "top_code_pymt_share",
                   "facility_share", "n_service_rows"]

providers = q(f"""
SELECT p.npi, p.provider_type, p.state, p.label,
       {", ".join("p." + c for c in feature_columns)},
       counts.n_in_specialty,
       CASE WHEN counts.n_in_specialty >= 30 THEN p.provider_type ELSE 'ALL' END AS peer_group
FROM providers p
JOIN (SELECT provider_type, COUNT(*) AS n_in_specialty FROM providers GROUP BY provider_type) counts
  USING (provider_type)
""")
print(providers.shape, " positives:", int(providers["label"].sum()))
print(providers["peer_group"].value_counts().head(8))

(20066, 22)  positives: 68
peer_group
Nurse Practitioner                               3143
Physician Assistant                              1705
Internal Medicine                                1434
Family Practice                                  1302
Physical Therapist in Private Practice           1218
Emergency Medicine                                807
Certified Registered Nurse Anesthetist (CRNA)     781
Anesthesiology                                    636
Name: count, dtype: int64


## 2. Engineer: robust peer z-scores

For each feature, within each peer group: subtract the median, divide by $1.4826 \times \text{MAD}$
(median absolute deviation). The constant makes MAD match the standard deviation for normal data.
Missing values become 0 ("no evidence either way"), and a MAD of zero falls back to the standard deviation.

$$z_{ij} = \frac{x_{ij} - \text{median}_{g(i)}(x_j)}{1.4826 \cdot \text{MAD}_{g(i)}(x_j)}$$

**What this cell does:** computes that in pandas with a groupby, and stores the per-group medians and
scales because the scoring function at the end needs them to score a new provider the same way.

In [3]:
def peer_statistics(frame, group_column, columns):
    """Per group and feature: median, and a scale = 1.4826 * MAD, falling back to std when MAD is 0."""
    stats = {}
    for group_name, group_rows in frame.groupby(group_column):
        for column in columns:
            values = group_rows[column].dropna()
            median = values.median()
            mad = (values - median).abs().median()
            scale = 1.4826 * mad if mad > 0 else values.std()
            stats[(group_name, column)] = (median, scale if scale and scale > 0 else 1.0)
    return stats

def apply_peer_zscores(frame, stats, group_column, columns):
    """z = (x - median_group) / scale_group, missing -> 0."""
    z = pd.DataFrame(index=frame.index)
    for column in columns:
        medians = frame[group_column].map(lambda g: stats[(g, column)][0])
        scales = frame[group_column].map(lambda g: stats[(g, column)][1])
        z["z_" + column] = ((frame[column] - medians) / scales).fillna(0.0)
    return z

stats = peer_statistics(providers, "peer_group", feature_columns)
Z = apply_peer_zscores(providers, stats, "peer_group", feature_columns).clip(-10, 10)   # clip extreme tails
z_columns = list(Z.columns)
print(Z.describe().T[["mean", "50%", "min", "max"]].round(2).head(6))

                          mean  50%    min   max
z_srvcs_per_bene          0.86  0.0  -6.53  10.0
z_pymt_per_bene           0.54  0.0  -6.24  10.0
z_chrg_to_alowd           0.47  0.0  -2.55  10.0
z_drug_pymt_share         0.31  0.0 -10.00  10.0
z_risk_score              0.31  0.0  -3.22  10.0
z_pymt_per_bene_per_risk  0.55  0.0  -4.43  10.0


## 3. Train: weighted, ridge-regularized logistic regression, from the derivations

**What this cell does:** stratified 5-fold cross-validation (68 positives cannot support one split),
gradient descent on the weighted ridge log loss from the top of the notebook, out-of-fold probabilities
for every provider. Then ranking metrics: AUC and precision at $k$ with $k = 100$, the investigator capacity.

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def fit_weighted_ridge_logistic(X, y, lam=0.01, learning_rate=0.2, n_steps=1500):
    """Gradient descent on weighted log loss + lam ||w||^2.  Returns (w, b)."""
    n_rows, n_features = X.shape
    n_positive = y.sum(); n_negative = n_rows - n_positive
    class_weight = np.where(y == 1, n_rows / (2 * n_positive), n_rows / (2 * n_negative))   # c_1, c_0
    w, b = np.zeros(n_features), 0.0
    for step in range(n_steps):
        p = sigmoid(X @ w + b)
        weighted_residual = class_weight * (p - y)                     # c (.) (p - y)
        grad_w = (X.T @ weighted_residual) / n_rows + 2 * lam * w      # + ridge term
        grad_b = np.sum(weighted_residual) / n_rows
        w -= learning_rate * grad_w
        b -= learning_rate * grad_b
    return w, b

def precision_at_k(scores, y, k):
    top = np.argsort(-scores)[:k]
    return y[top].mean()

X_all = Z.to_numpy(dtype=float)
y_all = providers["label"].to_numpy(dtype=float)
out_of_fold = np.zeros(len(y_all))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold_number, (train_index, test_index) in enumerate(folds.split(X_all, y_all), start=1):
    w_fold, b_fold = fit_weighted_ridge_logistic(X_all[train_index], y_all[train_index])
    out_of_fold[test_index] = sigmoid(X_all[test_index] @ w_fold + b_fold)
    print(f"fold {fold_number}: test positives {int(y_all[test_index].sum())}, AUC {roc_auc_score(y_all[test_index], out_of_fold[test_index]):.3f}")

print(f"\nout-of-fold AUC          : {roc_auc_score(y_all, out_of_fold):.3f}")
print(f"precision at 100         : {precision_at_k(out_of_fold, y_all, 100):.3f}   ({int(precision_at_k(out_of_fold, y_all, 100) * 100)} of the 68 in the top 100)")
print(f"base rate                : {y_all.mean():.4f}")

fold 1: test positives 14, AUC 0.645


fold 2: test positives 14, AUC 0.724


fold 3: test positives 14, AUC 0.593


fold 4: test positives 13, AUC 0.549


fold 5: test positives 13, AUC 0.630

out-of-fold AUC          : 0.626
precision at 100         : 0.000   (0 of the 68 in the top 100)
base rate                : 0.0034


## 4. Score and write back

A model that lives in a notebook variable is not a system. **What this cell does:** fits the final
weights on all rows, scores every provider, and writes a `provider_scores` table into the database
with the score, its rank, and the model version. The weights, intercept, and peer statistics are saved
to a JSON file so the scoring function can run without the training data.

In [5]:
import json
from datetime import date

w_final, b_final = fit_weighted_ridge_logistic(X_all, y_all)
scores = sigmoid(X_all @ w_final + b_final)

provider_scores = pd.DataFrame({
    "npi": providers["npi"],
    "score": scores,
    "rank": pd.Series(scores).rank(ascending=False, method="first").astype(int).values,
    "model_version": f"ridge_logit_v1_{date.today().isoformat()}",
})
provider_scores.to_sql("provider_scores", con, if_exists="replace", index=False)

artifact = {
    "model_version": provider_scores["model_version"].iloc[0],
    "feature_columns": feature_columns,
    "weights": dict(zip(z_columns, w_final.round(6).tolist())),
    "intercept": float(b_final),
    "peer_statistics": {f"{group}|{column}": [float(m), float(s)] for (group, column), (m, s) in stats.items()},
}
artifact_path = "C:/Users/palla/OneDrive/Documents/Coding Projects/cheatsheets_and_templates/ML_derivations/provider_model_v1.json"
with open(artifact_path, "w") as f:
    json.dump(artifact, f, indent=1)
print("wrote provider_scores table and", artifact_path)
print("\nlargest weights (positive = raises the score):")
print(pd.Series(artifact["weights"]).sort_values(key=abs, ascending=False).head(8).round(3))

wrote provider_scores table and C:/Users/palla/OneDrive/Documents/Coding Projects/cheatsheets_and_templates/ML_derivations/provider_model_v1.json

largest weights (positive = raises the score):
z_chrg_to_alowd            -0.428
z_pymt_per_bene            -0.276
z_pymt_per_bene_per_risk    0.247
z_n_distinct_codes         -0.231
z_pct_dual                  0.221
z_em_high_share             0.155
z_srvcs_per_bene            0.138
z_n_service_rows            0.109
dtype: float64


## 5. Serve: the worklist is a SQL query

**What this cell does:** the investigator-facing query. Joins the scores back to provider attributes
and returns the top 15 with their specialty and, since this is evaluation data, whether they were
a known exclusion. In production the `label` column would not exist yet; that is the point.

In [6]:
q("""
SELECT s.rank, s.npi, p.provider_type, p.state,
       ROUND(s.score, 4) AS score,
       p.label AS known_exclusion
FROM provider_scores s
JOIN providers p USING (npi)
ORDER BY s.rank
LIMIT 15
""")

,rank,npi,provider_type,state,score,known_exclusion
0,1,1528032737,Emergency Medicine,VA,0.9667,0
1,2,1417197435,General Surgery,DC,0.9655,0
2,3,1003969767,Opioid Treatment Program,IN,0.9601,0
3,4,1982847851,Plastic and Reconstructive Surgery,CA,0.9510,0
4,5,1366504250,Dermatology,CA,0.9496,0
5,6,1457886806,Physical Therapist in Private Practice,AK,0.9485,0
6,7,1699733766,Allergy/ Immunology,CA,0.9476,0
7,8,1427426386,Nurse Practitioner,CA,0.9451,0
8,9,1215986229,Diagnostic Radiology,CA,0.9422,0
9,10,1023863487,Nurse Practitioner,CA,0.9415,0


**Read the worklist honestly.** Out-of-fold AUC is about 0.63, and none of the 68 known exclusions
are in the top 100. That matches what the full Medicare project found with far stronger models: on
annual billing aggregates, the known positives sit modestly off-normal on several features and never
win a ranking contest against providers who are extreme on one.

There is also a design flaw visible in the table, and it is worth being able to spot it: the top of
the list is full of rare specialties (opioid treatment programs, thoracic surgery, plastic surgery).
Those fell below the 30-provider cutoff and were z-scored against the **global** median, so a
specialty whose normal billing is unlike everyone else's looks extreme on every feature. The model
learned "rare specialty" as a risk factor. The fix is a higher minimum group size, or dropping
tiny specialties from scoring, or a specialty indicator as a feature so the model can separate
"unusual specialty" from "unusual for the specialty." Finding that in a top-15 table is the kind
of check a reviewer expects before any list goes to an investigator.

## 6. Score a new provider from the saved artifact

**What this cell does:** a function that takes one provider's raw features and specialty, applies the
saved peer statistics and weights, and returns a probability. No training data, no pandas groupby:
this is the function a web service would call.

In [7]:
def score_new_provider(raw_features: dict, provider_type: str, artifact: dict) -> float:
    """raw_features: {feature_name: value}. Returns the model's probability for this provider."""
    logit = artifact["intercept"]
    for column in artifact["feature_columns"]:
        key = f"{provider_type}|{column}"
        if key not in artifact["peer_statistics"]:
            key = f"ALL|{column}"                                   # small specialty: global stats
        median, scale = artifact["peer_statistics"][key]
        value = raw_features.get(column)
        z = 0.0 if value is None else max(-10, min(10, (value - median) / scale))
        logit += artifact["weights"]["z_" + column] * z
    return 1.0 / (1.0 + np.exp(-logit))

with open(artifact_path) as f:
    loaded_artifact = json.load(f)

# A made-up internist: heavy per-patient billing, high E&M level share, dual-heavy panel.
new_provider = {"srvcs_per_bene": 9.5, "pymt_per_bene": 600.0, "chrg_to_alowd": 1.8, "drug_pymt_share": 0.0,
                "risk_score": 1.1, "pymt_per_bene_per_risk": 545.0, "pct_under65": 0.3, "pct_dual": 0.5,
                "avg_age": 70, "n_distinct_codes": 12, "tot_benes": 150, "sameday_repeat_ratio": 1.4,
                "em_high_share": 0.95, "top_code_pymt_share": 0.6, "facility_share": 0.0, "n_service_rows": 10}
print(f"score for the new internist: {score_new_provider(new_provider, 'Internal Medicine', loaded_artifact):.4f}")
print(f"for comparison, the median provider score is {np.median(scores):.4f} and the 99th percentile is {np.quantile(scores, 0.99):.4f}")

score for the new internist: 0.8535
for comparison, the median provider score is 0.4325 and the 99th percentile is 0.8493


## What makes this "production-shaped"

- Features come from **one SQL query** that could run on Snowflake or Databricks unchanged except the dialect.
- The peer statistics are **saved with the model**, so a new provider is scored exactly the way the training data was.
- Scores are **written to a table with a version stamp**, so the worklist is a query and the model that produced it is traceable.
- The scoring function **has no dependency on the training data**, which is what lets it run in a service.
- Evaluation was **out of fold with the metric that matches deployment** (precision at the investigator capacity).

The same shape runs the Medicare project's real pipeline; the only differences there are the model
(gradient boosting), the database (DuckDB), and the scale (1.3M providers).

In [8]:
con.close()